In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    kendalltau,
    theilslopes
)

from IPython.display import display


# Keep notebook output clean.
warnings.filterwarnings("ignore")


# Make DataFrames easier to inspect.
pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    100
)

pd.set_option(
    "display.width",
    140
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}"
)


print("=" * 80)
print("H5 ANALYSIS ENVIRONMENT READY")
print("=" * 80)

print("✓ Libraries imported")
print("✓ Statistical functions loaded")
print("✓ Display settings configured")

In [ ]:
# ============================================================
# H5 — ANALYSIS PARAMETERS
# ============================================================

ALPHA = 0.05

CANDIDATE_PATHS = [
    Path(r"C:\Users\bhara\OneDrive\Desktop\DAMO-6994-\DAMO-6994-\DAMO-6994-\data\cleaned dataset\Explanatory_and_Predictive_ED_Analytics_Dataset(2).xlsx"),
    Path(r"C:\Users\bhara\OneDrive\Desktop\DAMO-6994-Capstone-Project-DAMO-6994-\DAMO-6994-\Capstone_Project-DAMO-6994-\data\cleaned dataset\Explanatory_and_Predictive_ED_Analytics_Dataset(2).xlsx"),
    Path(r"C:\Users\bhara\OneDrive\Desktop\Capstone_Project-DAMO-6994-\Capstone_Project-DAMO-6994-\Capstone_Project-DAMO-6994-\data\cleaned dataset\Explanatory_and_Predictive_ED_Analytics_Dataset.xlsx"),
    Path(r"../../data/cleaned dataset/Explanatory_and_Predictive_ED_Analytics_Dataset.xlsx"),
    Path(r"data/cleaned dataset/Explanatory_and_Predictive_ED_Analytics_Dataset.xlsx")
]

INPUT_FILE = None
for candidate in CANDIDATE_PATHS:
    if candidate.exists():
        INPUT_FILE = candidate
        break

if INPUT_FILE is None:
    INPUT_FILE = CANDIDATE_PATHS[0]

SHEET_NAME = "Age_Sex"


print(
    f"Significance level: α = {ALPHA}"
)

print(
    f"Source sheet: {SHEET_NAME}"
)

In [ ]:
# ============================================================
# H5 — LOAD AGE_SEX SHEET
# ============================================================

age_sex_df = pd.read_excel(
    INPUT_FILE,
    sheet_name=SHEET_NAME
)


print("=" * 80)
print("H5 — AGE_SEX DATA LOADED")
print("=" * 80)

print(
    f"Rows loaded: {len(age_sex_df):,}"
)

print(
    "\nColumns:"
)

print(
    list(age_sex_df.columns)
)

display(
    age_sex_df.head()
)

In [ ]:
# ============================================================
# H5 — REQUIRED COLUMN VALIDATION
# ============================================================

REQUIRED_COLUMNS = [
    "fiscal_year",
    "fiscal_year_start",
    "sex",
    "age_group",
    "ed_visits",
    "median_los_minutes"
]


missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in age_sex_df.columns
]


if missing_columns:

    raise KeyError(
        "Missing required H5 columns: "
        +
        ", ".join(
            missing_columns
        )
    )


print(
    "✓ All required H5 columns are present."
)

In [ ]:
# ============================================================
# H5 — DATA CLEANING
# ============================================================

h5_df = (
    age_sex_df[
        REQUIRED_COLUMNS
    ]
    .copy()
)


h5_df["fiscal_year_start"] = pd.to_numeric(
    h5_df["fiscal_year_start"],
    errors="coerce"
)


h5_df["ed_visits"] = pd.to_numeric(
    h5_df["ed_visits"],
    errors="coerce"
)


h5_df["median_los_minutes"] = pd.to_numeric(
    h5_df["median_los_minutes"],
    errors="coerce"
)


h5_df = h5_df.dropna(
    subset=[
        "fiscal_year_start",
        "ed_visits",
        "median_los_minutes"
    ]
).copy()


h5_df = h5_df[
    (h5_df["ed_visits"] >= 0)
    &
    (h5_df["median_los_minutes"] >= 0)
].copy()


h5_df["fiscal_year_start"] = (
    h5_df["fiscal_year_start"]
    .astype(int)
)


print(
    f"Clean H5 records: {len(h5_df):,}"
)

In [ ]:
# ============================================================
# H5 — FISCAL YEAR VALIDATION
# ============================================================

fiscal_years = (
    sorted(
        h5_df[
            "fiscal_year_start"
        ]
        .unique()
        .tolist()
    )
)


print(
    "Fiscal years:"
)

print(
    fiscal_years
)


if len(fiscal_years) != 19:

    raise ValueError(
        "H5 requires exactly 19 fiscal-year observations, "
        f"but {len(fiscal_years)} fiscal years were found."
    )


expected_years = list(
    range(
        min(fiscal_years),
        max(fiscal_years) + 1
    )
)


if fiscal_years != expected_years:

    raise ValueError(
        "Fiscal years are not a complete consecutive "
        "19-year series."
    )


print(
    "\n✓ Exactly 19 consecutive fiscal years confirmed."
)

In [ ]:
# ============================================================
# H5 — AGE_SEX STRUCTURE VALIDATION
# ============================================================

strata_check = (
    h5_df
    .groupby(
        "fiscal_year_start",
        observed=True
    )
    .agg(
        Sex_Groups=(
            "sex",
            "nunique"
        ),

        Age_Groups=(
            "age_group",
            "nunique"
        ),

        Records=(
            "fiscal_year_start",
            "size"
        )
    )
    .reset_index()
)


display(
    strata_check
)


if not (
    strata_check["Records"] == 10
).all():

    raise ValueError(
        "Every fiscal year is expected to contain "
        "10 Age_Sex aggregate strata."
    )


print(
    "✓ Each fiscal year contains 10 Age_Sex strata."
)

In [ ]:
# ============================================================
# H5 — DUPLICATE CHECK
# ============================================================

duplicate_columns = [
    "fiscal_year_start",
    "sex",
    "age_group"
]


duplicate_mask = (
    h5_df
    .duplicated(
        subset=duplicate_columns,
        keep=False
    )
)


duplicate_count = (
    duplicate_mask.sum()
)


print(
    f"Duplicate fiscal-year × sex × age-group records: "
    f"{duplicate_count:,}"
)


if duplicate_count > 0:

    display(
        h5_df[
            duplicate_mask
        ]
        .sort_values(
            duplicate_columns
        )
    )

    raise ValueError(
        "Duplicate fiscal-year × sex × age-group "
        "records were detected."
    )


print(
    "✓ No duplicate demographic strata found."
)

In [ ]:
# ============================================================
# H5 — ESTIMATED RESOURCE BURDEN INDEX
# ============================================================

h5_df["ERBI"] = (
    h5_df["ed_visits"]
    *
    h5_df["median_los_minutes"]
)


print(
    "✓ ERBI calculated."
)


display(
    h5_df[
        [
            "fiscal_year_start",
            "sex",
            "age_group",
            "ed_visits",
            "median_los_minutes",
            "ERBI"
        ]
    ]
    .head()
)

In [ ]:
# ============================================================
# H5 — ANNUAL ERBI SERIES
# ============================================================

annual_erbi = (
    h5_df
    .groupby(
        "fiscal_year_start",
        observed=True
    )
    .agg(
        Annual_ERBI=(
            "ERBI",
            "sum"
        ),

        Annual_ED_Visits=(
            "ed_visits",
            "sum"
        ),

        Mean_Reported_Median_LOS=(
            "median_los_minutes",
            "mean"
        ),

        Median_Reported_Median_LOS=(
            "median_los_minutes",
            "median"
        ),

        Demographic_Strata=(
            "age_group",
            "size"
        )
    )
    .reset_index()
    .sort_values(
        "fiscal_year_start"
    )
    .reset_index(
        drop=True
    )
)


display(
    annual_erbi.round(2)
)

In [ ]:
# ============================================================
# H5 — ANNUAL SERIES VALIDATION
# ============================================================

if len(annual_erbi) != 19:

    raise ValueError(
        "Annual ERBI must contain exactly 19 fiscal-year "
        f"observations, but {len(annual_erbi)} were found."
    )


if annual_erbi[
    "Annual_ERBI"
].isna().any():

    raise ValueError(
        "Missing annual ERBI values detected."
    )


if not (
    annual_erbi[
        "Demographic_Strata"
    ] == 10
).all():

    raise ValueError(
        "Every fiscal year must contain exactly "
        "10 Age_Sex strata."
    )


print("=" * 80)
print("H5 ANNUAL SERIES VALIDATED")
print("=" * 80)

print(
    f"Fiscal-year observations: {len(annual_erbi)}"
)

print(
    "✓ No missing annual ERBI values"
)

print(
    "✓ 10 demographic strata per fiscal year"
)

In [ ]:
# ============================================================
# H5 — MANN-KENDALL MONOTONIC TREND TEST
# ============================================================

years = (
    annual_erbi[
        "fiscal_year_start"
    ]
    .to_numpy()
)


erbi_values = (
    annual_erbi[
        "Annual_ERBI"
    ]
    .to_numpy()
)


kendall_tau, p_value = kendalltau(
    years,
    erbi_values,
    alternative="two-sided"
)


print("=" * 80)
print("H5 — MANN-KENDALL TREND TEST")
print("=" * 80)

print(
    f"Kendall's τ: {kendall_tau:.6f}"
)

print(
    f"P-value: {p_value:.10g}"
)

print(
    f"Alpha: {ALPHA}"
)

In [ ]:
# ============================================================
# H5 — TREND DIRECTION
# ============================================================

if kendall_tau > 0:

    trend_direction = "Positive / Increasing"

elif kendall_tau < 0:

    trend_direction = "Negative / Decreasing"

else:

    trend_direction = "No Monotonic Direction"


print(
    f"Trend direction: {trend_direction}"
)

In [ ]:
# ============================================================
# H5 — HYPOTHESIS DECISION
# ============================================================

if p_value < ALPHA:

    decision = "Reject H₀"

    if kendall_tau > 0:

        conclusion = (
            "There is statistically significant evidence "
            "of a positive monotonic trend in the estimated "
            "ED resource-burden proxy across the 19 fiscal years."
        )

    elif kendall_tau < 0:

        conclusion = (
            "There is statistically significant evidence "
            "of a negative monotonic trend in the estimated "
            "ED resource-burden proxy across the 19 fiscal years."
        )

    else:

        conclusion = (
            "The trend test is statistically significant, "
            "but Kendall's tau is approximately zero."
        )

else:

    decision = "Fail to Reject H₀"

    conclusion = (
        "There is insufficient statistical evidence "
        "of a statistically significant monotonic trend "
        "in the estimated ED resource-burden proxy "
        "across the 19 fiscal years."
    )


print(
    f"Decision: {decision}"
)

print(
    f"\nConclusion:\n{conclusion}"
)

In [ ]:
# ============================================================
# H5 — SEN'S SLOPE
# ============================================================

sen_result = theilslopes(
    erbi_values,
    years,
    alpha=0.95
)


sen_slope = (
    sen_result.slope
)

sen_intercept = (
    sen_result.intercept
)

sen_lower = (
    sen_result.low_slope
)

sen_upper = (
    sen_result.high_slope
)


print("=" * 80)
print("H5 — SEN'S SLOPE")
print("=" * 80)

print(
    f"Sen's slope: {sen_slope:,.4f}"
)

print(
    f"95% lower slope: {sen_lower:,.4f}"
)

print(
    f"95% upper slope: {sen_upper:,.4f}"
)

In [ ]:
# ============================================================
# H5 — DESCRIPTIVE LONG-RUN CHANGE
# ============================================================

first_erbi = (
    annual_erbi.iloc[0][
        "Annual_ERBI"
    ]
)

last_erbi = (
    annual_erbi.iloc[-1][
        "Annual_ERBI"
    ]
)


absolute_change = (
    last_erbi
    -
    first_erbi
)


if first_erbi != 0:

    percentage_change = (
        absolute_change
        /
        first_erbi
        *
        100
    )

else:

    percentage_change = np.nan


print(
    f"First fiscal year ERBI: {first_erbi:,.0f}"
)

print(
    f"Last fiscal year ERBI: {last_erbi:,.0f}"
)

print(
    f"Absolute ERBI change: {absolute_change:,.0f}"
)

print(
    f"Percentage ERBI change: {percentage_change:.2f}%"
)

In [ ]:
# ============================================================
# H5 VISUAL 1
# ANNUAL ERBI TREND
# ============================================================

plt.figure(
    figsize=(13, 7)
)


plt.plot(
    annual_erbi[
        "fiscal_year_start"
    ],

    annual_erbi[
        "Annual_ERBI"
    ],

    marker="o",

    linewidth=2
)


plt.title(
    "Estimated ED Resource-Burden Proxy Across Fiscal Years",
    fontsize=15,
    fontweight="bold"
)


plt.xlabel(
    "Fiscal Year Start"
)


plt.ylabel(
    "Estimated Resource Burden Index (ERBI)"
)


plt.xticks(
    years,
    rotation=45
)


plt.grid(
    axis="y",
    alpha=0.25
)


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# H5 VISUAL 2
# ERBI WITH SEN'S SLOPE
# ============================================================

fitted_erbi = (
    sen_intercept
    +
    sen_slope
    *
    years
)


plt.figure(
    figsize=(13, 7)
)


plt.plot(
    years,
    erbi_values,
    marker="o",
    linewidth=2,
    label="Annual ERBI"
)


plt.plot(
    years,
    fitted_erbi,
    linestyle="--",
    linewidth=2,
    label="Sen's slope trend"
)


plt.title(
    "Estimated ED Resource-Burden Proxy and Monotonic Trend",
    fontsize=15,
    fontweight="bold"
)


plt.xlabel(
    "Fiscal Year Start"
)


plt.ylabel(
    "Estimated Resource Burden Index (ERBI)"
)


plt.xticks(
    years,
    rotation=45
)


plt.legend()


plt.grid(
    axis="y",
    alpha=0.25
)


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# H5 VISUAL 3
# ANNUAL ED VISIT VOLUME
# ============================================================

plt.figure(
    figsize=(13, 6)
)


plt.bar(
    annual_erbi[
        "fiscal_year_start"
    ],

    annual_erbi[
        "Annual_ED_Visits"
    ]
)


plt.title(
    "Annual Aggregate ED Visit Volume",
    fontsize=15,
    fontweight="bold"
)


plt.xlabel(
    "Fiscal Year Start"
)


plt.ylabel(
    "Aggregate ED Visits"
)


plt.xticks(
    years,
    rotation=45
)


plt.grid(
    axis="y",
    alpha=0.25
)


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# H5 — OPTIONAL EXPLORATORY TWO-YEAR FORECAST
# ============================================================

from scipy.stats import linregress


forecast_fit = linregress(
    years,
    erbi_values
)


forecast_slope = (
    forecast_fit.slope
)

forecast_intercept = (
    forecast_fit.intercept
)

forecast_r_squared = (
    forecast_fit.rvalue ** 2
)


future_years = np.array(
    [
        years[-1] + 1,
        years[-1] + 2
    ]
)


forecast_values = (
    forecast_intercept
    +
    forecast_slope
    *
    future_years
)


forecast_table = pd.DataFrame({

    "Fiscal_Year_Start": future_years,

    "Exploratory_ERBI_Forecast": forecast_values

})


display(
    forecast_table.round(2)
)


print(
    f"Linear benchmark R²: "
    f"{forecast_r_squared:.4f}"
)


print(
    "\nForecast interpretation:"
)

print(
    "This is an exploratory benchmark based on only "
    "19 annual observations. It should not be interpreted "
    "as a precise prediction of future ED resource utilization."
)

In [ ]:
# ============================================================
# H5 — FINAL RESULTS TABLE
# ============================================================

h5_results = pd.DataFrame({

    "Hypothesis": [
        "H5"
    ],

    "Research_Question": [
        "Has the estimated ED resource-burden proxy shown a statistically significant monotonic trend across the 19 fiscal years?"
    ],

    "Outcome": [
        "Estimated Resource Burden Index (ERBI)"
    ],

    "Annual_Observations": [
        len(annual_erbi)
    ],

    "Statistical_Test": [
        "Mann-Kendall trend test"
    ],

    "Kendall_Tau": [
        kendall_tau
    ],

    "P_Value": [
        p_value
    ],

    "Alpha": [
        ALPHA
    ],

    "Sen_Slope": [
        sen_slope
    ],

    "Sen_Lower_Slope": [
        sen_lower
    ],

    "Sen_Upper_Slope": [
        sen_upper
    ],

    "Trend_Direction": [
        trend_direction
    ],

    "Decision": [
        decision
    ]
})


display(
    h5_results.round(6)
)

In [ ]:
# ============================================================
# H5 — FINAL INTERPRETATION
# ============================================================

print("=" * 90)
print("H5 — FINAL HYPOTHESIS TEST RESULT")
print("=" * 90)


print(
    "\nResearch Question:"
)

print(
    "Has the estimated ED resource-burden proxy shown "
    "a statistically significant monotonic trend across "
    "the 19 fiscal years?"
)


print(
    "\nStatistical Test:"
)

print(
    "Mann-Kendall trend test"
)


print(
    f"\nKendall's τ = "
    f"{kendall_tau:.6f}"
)


print(
    f"p = "
    f"{p_value:.10g}"
)


print(
    f"α = "
    f"{ALPHA}"
)


print(
    f"Trend direction = "
    f"{trend_direction}"
)


print(
    f"Sen's slope = "
    f"{sen_slope:,.4f} ERBI units per fiscal year"
)


print(
    f"\nDecision: "
    f"{decision}"
)


print(
    f"\nConclusion:\n"
    f"{conclusion}"
)


print(
    "\nMethodological limitation:"
)


print(
    "ERBI is an aggregate resource-burden proxy calculated "
    "as ED visit volume multiplied by reported median LOS. "
    "It is not equivalent to actual total ED minutes. "
    "The 19 annual observations are aggregate fiscal-year "
    "measurements. Therefore, the findings should not be "
    "interpreted as patient-level effects or causal relationships."
)

In [ ]:
# ============================================================
# H5 — VARIABLE SAFETY CHECK
# ============================================================

required_h5_variables = [
    "h5_df",
    "annual_erbi",
    "years",
    "erbi_values",
    "kendall_tau",
    "p_value",
    "ALPHA",
    "trend_direction",
    "decision",
    "conclusion",
    "sen_slope",
    "sen_lower",
    "sen_upper",
    "h5_results"
]


missing_variables = [
    variable
    for variable in required_h5_variables
    if variable not in globals()
]


if missing_variables:

    raise NameError(
        "The following required H5 variables are missing: "
        +
        ", ".join(
            missing_variables
        )
    )


print("=" * 80)
print("ALL REQUIRED H5 VARIABLES ARE DEFINED")
print("=" * 80)


for variable in required_h5_variables:

    print(
        f"  ✓ {variable}"
    )

In [ ]:
# ============================================================
# H5 — FINAL METHODOLOGY AUDIT
# ============================================================

print("=" * 80)
print("H5 FINAL METHODOLOGY AUDIT")
print("=" * 80)

print("✓ Outcome: Estimated Resource Burden Index (ERBI)")
print("✓ Time series: 19 fiscal years")
print("✓ Test: Two-sided Mann-Kendall")
print("✓ Effect/trend measure: Kendall's tau")
print("✓ Trend magnitude: Sen's slope")
print("✓ Aggregate-level interpretation")
print("✓ No patient-level claims")
print("✓ No causal claims")
print("✓ No ED-visit frequency weighting")
print("✓ No weighted Kruskal-Wallis")
print("✓ No weighted Dunn")
print("✓ No actual Total ED-Minutes claim")
print("✓ No Excel output")
print("✓ No CSV output")

### Q&A
* **Has the estimated ED resource-burden proxy shown a statistically significant monotonic trend across the 19 fiscal years?**
  Yes. The two-sided Mann-Kendall monotonic trend test yielded a Kendall's $\tau = 0.9766$ ($p = 3.1074 \times 10^{-15} < 0.05$), rejecting the null hypothesis ($H_0$).

### Data Analysis Key Findings
* **Sample Structure**: Exactly 19 consecutive fiscal years (2003–2021) constructed from 190 aggregate strata (10 sex $\times$ age-group strata per fiscal year).
* **Estimated Resource Burden Index (ERBI)**: The non-parametric Sen's slope estimate indicates an annual increase of approximately **119,140,157 ERBI proxy units per fiscal year** (95% confidence interval: [103,256,543, 151,985,469]).
* **Long-Run Proxy Change**: Annual ERBI increased from 651,859,054 in FY 2003 to 3,190,225,055 in FY 2021, representing an overall descriptive increase of +389.40% across the 19 fiscal years.
* **Exploratory Benchmark Forecast**: A simple linear trend benchmark ($R^2 = 0.9575$) projects exploratory benchmark ERBI levels of $3,588,683,678$ (Year +1) and $3,722,897,419$ (Year +2).

### Insights or Next Steps
* The monotonic expansion of the aggregate ERBI proxy reflects compounding systemic pressures from increasing ED visit counts and lengthening median duration of stay.
* All findings are aggregate-level proxy evaluations and must not be interpreted as causal or patient-level total minutes.